# EXP-07: temperature scaling + threshold tuning on P10 checkpoints
No retraining. Per-branch T fit on VAL (BCE/LBFGS); operating threshold swept on VAL (max DSC_pos). Test reported at tuned point + 0.5 ref. det=seed42, ens=mean(42,43,44). MC skipped (P10-degenerate).

In [ ]:

import os, subprocess
_q = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True)
_sm = _q.stdout.strip().split(".")
SM = (int(_sm[0]), int(_sm[1])) if len(_sm) == 2 and _sm[0].strip().isdigit() else (9, 0)
print("GPU SM:", SM)
if SM < (7, 0):
    subprocess.run(["pip", "install", "-q", "torch==2.3.1+cu118", "torchvision==0.18.1+cu118",
                    "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
os.system("pip install -q pydicom albumentations pretrainedmodels efficientnet_pytorch tqdm munch scikit-learn")
os.system("pip install -q --no-deps segmentation-models-pytorch")
import sys, time, math, glob, ast, shutil, json, random
import numpy as np, pandas as pd, pydicom, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Tuple, Optional, Any
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
print("torch:", torch.__version__, "| smp:", smp.__version__, "| alb:", A.__version__)
device = torch.device("cuda")
seed_everything = lambda s=42: (random.seed(s), np.random.seed(s), torch.manual_seed(s),
                                torch.cuda.manual_seed_all(s))


In [ ]:

df_splits = pd.read_csv(glob.glob("/kaggle/input/**/patient_splits.csv", recursive=True)[0])
dmap0 = {}
for p in glob.glob("/kaggle/input/**/dicom-images-train/**/*.dcm", recursive=True):
    dmap0.setdefault(os.path.basename(p).replace(".dcm", ""), p)
df_splits["dcm_path"] = df_splits["ImageId"].map(dmap0)
print(len(df_splits), "missing:", int(df_splits["dcm_path"].isna().sum()))


In [ ]:
# 3. SIIM-ACR RLE Decoder with Multi-Mask Logical OR Aggregation

def rle_decode(rle_str: Any, shape: tuple = (1024, 1024)) -> np.ndarray:
    """Decode SIIM-ACR Run-Length Encoded string into binary mask (Fortran order)."""
    if rle_str is None or (isinstance(rle_str, float) and np.isnan(rle_str)):
        return np.zeros(shape, dtype=np.uint8)

    rle_str = str(rle_str).strip()
    if rle_str == "" or rle_str == "-1":
        return np.zeros(shape, dtype=np.uint8)

    s = rle_str.split()
    starts = np.asarray([int(float(x)) for x in s[0::2]], dtype=int) - 1
    lengths = np.asarray([int(float(x)) for x in s[1::2]], dtype=int)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order="F")

def aggregate_rle_list(rle_list: List[str], shape: tuple = (1024, 1024)) -> np.ndarray:
    """Combine multiple RLE instances for an image using logical OR."""
    composite = np.zeros(shape, dtype=np.uint8)
    for rle in rle_list:
        mask = rle_decode(rle, shape=shape)
        composite = np.bitwise_or(composite, mask)
    return composite

# Verify RLE decoder on sample
sample_rles = ast.literal_eval(df_splits.iloc[0]["EncodedPixelsList"])
sample_mask = aggregate_rle_list(sample_rles)
print(f"Verification: sample mask shape={sample_mask.shape}, sum={np.sum(sample_mask)}")


In [ ]:
# 4. PyTorch Dataset and Radiologically Sound Augmentations

class SIIMPneumothoraxDataset(Dataset):
    """Dataset for SIIM-ACR pneumothorax radiographs with dense masks."""

    def __init__(self, df: pd.DataFrame, image_size: int = 512, transforms: Optional[Any] = None):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        dcm_path = row["dcm_path"]
        
        # Read DICOM
        try:
            dcm = pydicom.dcmread(dcm_path)
            img = dcm.pixel_array.astype(np.float32)
            if hasattr(dcm, "PhotometricInterpretation") and dcm.PhotometricInterpretation == "MONOCHROME1":
                img = np.amax(img) - img
        except Exception:
            img = np.zeros((1024, 1024), dtype=np.float32)

        # Normalize to uint8 [0, 255]
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = ((img - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
        else:
            img = np.zeros_like(img, dtype=np.uint8)

        # Grayscale to 3-channel
        img_3c = np.repeat(np.expand_dims(img, axis=-1), 3, axis=-1)

        # Decode ground-truth mask
        rle_raw = row["EncodedPixelsList"]
        if isinstance(rle_raw, str):
            try:
                rle_list = ast.literal_eval(rle_raw)
            except Exception:
                rle_list = [rle_raw]
        else:
            rle_list = [str(rle_raw)]

        mask = aggregate_rle_list(rle_list, shape=img.shape)

        # Apply Albumentations transforms
        if self.transforms is not None:
            augmented = self.transforms(image=img_3c, mask=mask)
            image_tensor = augmented["image"]
            mask_tensor = augmented["mask"].unsqueeze(0).float()
        else:
            image_tensor = torch.from_numpy(img_3c).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "image_id": row["ImageId"],
            "patient_id": str(row["PatientID"]),
            "view_position": str(row["ViewPosition"]),
            "has_pneumothorax": int(row["HasPneumothorax"])
        }

# Data augmentations: HorizontalFlip, ShiftScaleRotate, RandomBrightnessContrast
# Strictly NO VerticalFlip (violates anatomical cephalocaudal orientation)
train_transforms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-10, 10), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Partition DataFrames
# Train: Folds 1, 2, 3, 4 | Val: Fold 0 | Test Holdout: Fold 'test'
train_df = df_splits[df_splits["Fold"].isin(["1", "2", "3", "4", 1, 2, 3, 4])].reset_index(drop=True)
val_df = df_splits[df_splits["Fold"].isin(["0", 0])].reset_index(drop=True)
test_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)

print(f"Train set: {len(train_df)} images ({train_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Validation set: {len(val_df)} images ({val_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Test Holdout set: {len(test_df)} images ({test_df['HasPneumothorax'].mean()*100:.1f}% positive)")

# Verify zero patient leakage
train_val_patients = set(train_df["PatientID"]).union(set(val_df["PatientID"]))
test_patients = set(test_df["PatientID"])
overlap = train_val_patients.intersection(test_patients)
assert len(overlap) == 0, f"DATA LEAKAGE DETECTED: {len(overlap)} overlapping patients!"
print(f"LEAKAGE PROOF VERIFIED: Exactly {len(overlap)} overlapping patients between train/val and test.")


In [ ]:
# 5. Architecture: ResNet34 U-Net with Spatial Dropout and Combined Loss
import segmentation_models_pytorch as smp

class PneumothoraxUNet(nn.Module):
    """ResNet34 U-Net supporting deterministic inference and MC Dropout."""

    def __init__(self, dropout_rate: float = 0.2, pretrained: bool = True):
        super().__init__()
        self.dropout_rate = dropout_rate
        weights = "imagenet" if pretrained else None
        
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            decoder_channels=(256, 128, 64, 32, 16)
        )
        
        # Inject SpatialDropout2d into decoder blocks for MC Dropout
        if dropout_rate > 0.0:
            for idx in range(len(self.model.decoder.blocks)):
                self.model.decoder.blocks[idx].add_module(
                    "spatial_dropout", nn.Dropout2d(p=dropout_rate)
                )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    @torch.no_grad()
    def predict_deterministic(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Deterministic forward pass (eval mode)."""
        self.eval()
        logits = self.forward(x)
        prob = torch.sigmoid(logits)
        eps = 1e-7
        p_clamped = torch.clamp(prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        return {"prob": prob, "entropy": entropy}

    @torch.no_grad()
    def predict_mc_dropout(self, x: torch.Tensor, num_samples: int = 20) -> Dict[str, torch.Tensor]:
        """Monte Carlo Dropout inference with T stochastic passes."""
        self.train() # Activates Spatial Dropout during inference
        samples = []
        for _ in range(num_samples):
            logits = self.forward(x)
            samples.append(torch.sigmoid(logits))
        
        stacked = torch.stack(samples, dim=0) # (T, B, 1, H, W)
        mean_prob = torch.mean(stacked, dim=0)
        variance = torch.var(stacked, dim=0, unbiased=True)
        
        eps = 1e-7
        p_clamped = torch.clamp(mean_prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        
        return {
            "mean": mean_prob,
            "variance": variance,
            "entropy": entropy
        }

# Combined Loss: 0.5 * BCE + 0.5 * SoftDice
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        cardinality = (probs.pow(2) + targets.pow(2)).sum()
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice

class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice_loss = SoftDiceLoss(smooth=1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_bce = F.binary_cross_entropy_with_logits(logits, targets)
        loss_dice = self.dice_loss(logits, targets)
        return self.bce_weight * loss_bce + self.dice_weight * loss_dice

criterion = CombinedBCEDiceLoss()
print("Model architecture and CombinedBCEDiceLoss verified.")


In [ ]:
# 7. Evaluation Metrics: Disaggregated Dice, ESCE, AUROC-ED, and AURC

def compute_dice_coefficient(y_true: np.ndarray, y_pred: np.ndarray, empty_score: float = 1.0) -> float:
    y_true_sum = np.sum(y_true)
    y_pred_sum = np.sum(y_pred)
    if y_true_sum == 0 and y_pred_sum == 0:
        return empty_score
    if y_true_sum == 0 or y_pred_sum == 0:
        return 0.0
    intersection = np.sum(y_true * y_pred)
    return float(2.0 * intersection / (y_true_sum + y_pred_sum))

def compute_segmentation_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_pred_prob >= threshold).astype(np.uint8)
    y_true = (y_true > 0).astype(np.uint8)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    is_positive = float(np.sum(y_true) > 0)
    dice = compute_dice_coefficient(y_true, y_pred)
    iou = float(tp / (tp + fp + fn)) if (tp + fp + fn) > 0 else (1.0 if is_positive == 0 else 0.0)
    sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else (1.0 if np.sum(y_pred) == 0 else 0.0)

    return {
        "dice": dice,
        "iou": iou,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "is_positive": is_positive,
    }

def compute_esce(probs: np.ndarray, targets: np.ndarray, n_bins: int = 10):
    probs_flat = probs.flatten()
    targets_flat = targets.flatten()
    total = len(probs_flat)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_confs = np.zeros(n_bins)
    bin_accs = np.zeros(n_bins)
    bin_counts = np.zeros(n_bins)
    esce = 0.0

    for i in range(n_bins):
        low, high = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (probs_flat > low) & (probs_flat <= high) if i > 0 else (probs_flat >= low) & (probs_flat <= high)
        count = np.sum(mask)
        bin_counts[i] = count
        if count > 0:
            conf = np.mean(probs_flat[mask])
            acc = np.mean(targets_flat[mask])
            bin_confs[i] = conf
            bin_accs[i] = acc
            esce += (count / total) * np.abs(acc - conf)

    return float(esce), bin_confs, bin_accs, bin_counts

def compute_brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs - targets) ** 2))

def compute_error_detection_auroc(uncertainty_map: np.ndarray, probs: np.ndarray, targets: np.ndarray, threshold: float = 0.5) -> float:
    preds = (probs >= threshold).astype(np.uint8)
    error = np.abs(targets - preds).flatten()
    uncertainty = uncertainty_map.flatten()
    if np.all(error == 0) or np.all(error == 1):
        return 0.5
    if len(error) > 100_000:
        idx = np.random.choice(len(error), size=100_000, replace=False)
        error = error[idx]
        uncertainty = uncertainty[idx]
    try:
        return float(roc_auc_score(error, uncertainty))
    except Exception:
        return 0.5

def aggregate_case_uncertainty(uncertainty_map: np.ndarray, k: int = 500) -> float:
    flat = uncertainty_map.flatten()
    k = min(k, len(flat))
    top_k_vals = np.partition(flat, -k)[-k:]
    return float(np.mean(top_k_vals))

def compute_risk_coverage_curve(case_uncertainties: np.ndarray, case_risks: np.ndarray, steps: int = 50, min_cov: float = 0.20):
    n = len(case_uncertainties)
    sorted_idx = np.argsort(case_uncertainties)
    coverages = np.linspace(min_cov, 1.0, steps)
    risks = np.zeros(steps)
    for i, cov in enumerate(coverages):
        retain_n = max(1, int(round(cov * n)))
        retained = sorted_idx[:retain_n]
        risks[i] = float(np.mean(case_risks[retained]))
    return coverages, risks

def compute_aurc(coverages: np.ndarray, risks: np.ndarray) -> float:
    cov_norm = (coverages - coverages[0]) / (coverages[-1] - coverages[0])
    try:
        from scipy.integrate import trapezoid
        return float(trapezoid(risks, cov_norm))
    except ImportError:
        trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
        return float(trapz_fn(risks, cov_norm))


In [ ]:

# ---- models (arch identical to P10) ----
class DetNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
    def forward(self, x): return self.model(x)

_pt_cands = glob.glob("/kaggle/input/**/*.pt", recursive=True)
print("Discovered .pt files:", _pt_cands)
wmap = {os.path.basename(p).replace(".pt", ""): p for p in _pt_cands}
print("Weights map keys:", sorted(wmap.keys()))
if not ({"det_seed42", "det_seed43", "det_seed44"} <= set(wmap)):
    print("ALL items in /kaggle/input:", glob.glob("/kaggle/input/**", recursive=True)[:100])
assert {"det_seed42", "det_seed43", "det_seed44"} <= set(wmap), "P10 weights missing"
def load_det(seed):
    m = DetNet().to(device)
    m.load_state_dict(torch.load(wmap[f"det_seed{seed}"], map_location=device)); m.eval()
    return m
nets = {s: load_det(s) for s in (42, 43, 44)}

F_ = lambda v: df_splits["Fold"].isin([v, int(v)])
va_df = df_splits[F_("0")].reset_index(drop=True)
te_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)
print(len(va_df), len(te_df))
assert set(va_df["PatientID"]).isdisjoint(set(te_df["PatientID"]))
va_ld = DataLoader(SIIMPneumothoraxDataset(va_df, image_size=512, transforms=val_transforms),
                   batch_size=1, shuffle=False, num_workers=2)
te_ld = DataLoader(SIIMPneumothoraxDataset(te_df, image_size=512, transforms=val_transforms),
                   batch_size=1, shuffle=False, num_workers=2)

NV, S = len(va_df), 512
# cache val masks once (uint8)
vmask = np.memmap("/tmp/val_masks.dat", dtype=np.uint8, mode="w+", shape=(NV, S, S))
for j, b in enumerate(va_ld):
    vmask[j] = (b["mask"].numpy()[0, 0] > 0).astype(np.uint8)
    if (j + 1) % 400 == 0: print(f"masks {j+1}/{NV}", flush=True)
vmask.flush()
vpos = np.array([m.sum() > 0 for m in vmask])
print("val pos:", int(vpos.sum()))

def dice_at(p, y, t):
    pb = (p >= t)
    if y.sum() == 0 and pb.sum() == 0: return 1.0
    if y.sum() == 0 or pb.sum() == 0: return 0.0
    return float(2 * np.logical_and(y > 0, pb).sum() / (y.sum() + pb.sum()))

def stream_val_logits(member_seeds):
    """Single-seed val logits -> f16 memmap (reused file)."""
    assert len(member_seeds) == 1
    s = member_seeds[0]; net = nets[s]
    lg = np.memmap("/tmp/val_logits.dat", dtype=np.float16, mode="w+", shape=(NV, S, S))
    with torch.no_grad():
        for j, b in enumerate(va_ld):
            pr = torch.sigmoid(net(b["image"].to(device)))[0, 0].cpu().numpy().astype(np.float32)
            lg[j] = np.log(pr / np.clip(1 - pr, 1e-7, 1)).astype(np.float16)
            if (j + 1) % 400 == 0: print(f"val {s} {j+1}/{NV}", flush=True)
    lg.flush()
    return lg

def stream_val_meanprob():
    lg = np.memmap("/tmp/val_logits.dat", dtype=np.float16, mode="w+", shape=(NV, S, S))
    mean = np.zeros((NV, S, S), dtype=np.float32)
    for k, s in enumerate((42, 43, 44)):
        net = nets[s]
        with torch.no_grad():
            for j, b in enumerate(va_ld):
                pr = torch.sigmoid(net(b["image"].to(device)))[0, 0].cpu().numpy().astype(np.float32)
                mean[j] += pr / 3.0
                if (j + 1) % 400 == 0: print(f"val ens {s} {j+1}/{NV}", flush=True)
    eps = 1e-7
    lg[:] = np.log(np.clip(mean, eps, 1 - eps) / np.clip(1 - mean, eps, 1 - eps)).astype(np.float16)
    lg.flush()
    return lg

def fit_temperature(lg):
    z = lg[:].reshape(-1).astype(np.float32)[::64]
    y = vmask[:].reshape(-1).astype(np.float32)[::64]
    zt = torch.from_numpy(z); yt = torch.from_numpy(y)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], max_iter=25, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(zt / torch.exp(logT), yt)
        loss.backward()
        return loss
    opt.step(closure)
    T = float(torch.exp(logT).item())
    print("T =", round(T, 4), flush=True)
    return T

def sweep_threshold(lg, T, tag):
    P = 1 / (1 + np.exp(-lg[:].astype(np.float32) / T))  # (NV,512,512) f32 = 1.8GB transient
    ths = [round(v, 2) for v in np.arange(0.05, 0.91, 0.05)]
    scored = {}
    for t in ths:
        dp = np.array([dice_at(P[j], vmask[j], t) for j in range(NV)])
        scored[t] = (float(dp[vpos].mean()), float(dp.mean()))
        print(f"{tag} t={t:.2f} val_pos={scored[t][0]:.4f} val_all={scored[t][1]:.4f}", flush=True)
    t0 = max(ths, key=lambda t: scored[t][0])
    for t in [round(t0 + d, 2) for d in (-0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.04) if 0.02 < t0 + d < 0.98]:
        dp = np.array([dice_at(P[j], vmask[j], t) for j in range(NV)])
        scored[t] = (float(dp[vpos].mean()), float(dp.mean()))
    best = max(scored, key=lambda t: scored[t][0])
    print(tag, "BEST t =", best, scored[best], flush=True)
    del P
    return best, scored

tuned = {}
curves = {}
# det branch
lg = stream_val_logits((42,))
T_det = fit_temperature(lg)
t_det, curves["det"] = sweep_threshold(lg, T_det, "det")
tuned["det"] = {"T": T_det, "t": t_det}
# ens branch
lg = stream_val_meanprob()
T_ens = fit_temperature(lg)
t_ens, curves["ens"] = sweep_threshold(lg, T_ens, "ens")
tuned["ens"] = {"T": T_ens, "t": t_ens}
print(json.dumps(tuned))

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)
for tag, col in (("det", "#e74c3c"), ("ens", "#2980b9")):
    xs = sorted(curves[tag]); ax.plot(xs, [curves[tag][t][0] for t in xs], color=col, label=f"{tag} val DSC_pos")
    ax.plot(xs, [curves[tag][t][1] for t in xs], color=col, linestyle="--", label=f"{tag} val DSC_all")
    ax.axvline(tuned[tag]["t"], color=col, linestyle=":")
ax.set_xlabel("Threshold", fontweight="bold"); ax.set_ylabel("Val Dice", fontweight="bold")
ax.set_title("EXP-07 val threshold sweep (star = selected)", fontweight="bold"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("exp07_threshold_curves.png", dpi=300); plt.close()

# ---- test @ tuned + @ 0.5 ----
def entropy(p): return -(p * np.log2(p + 1e-8) + (1 - p) * np.log2(1 - p + 1e-8))
def topk(a, k=500):
    f = a.ravel(); return float(f[np.argpartition(f, -k)[-k:]].mean())
N = len(te_df); B = 10
acc = {k: {"n": 0, "bn": np.zeros(B), "bp": np.zeros(B), "by": np.zeros(B), "br": 0.0}
       for k in ("det", "det05", "ens", "ens05")}
conf = {k: {"tp": 0, "fp": 0, "fn": 0, "tn": 0} for k in ("det", "det05", "ens", "ens05")}
def acc_up(a, p, y):
    b = np.clip((p * B).astype(int), 0, B - 1); a["n"] += y.size
    for i in range(B):
        m = b == i
        if m.any(): a["bn"][i] += m.sum(); a["bp"][i] += p[m].sum(); a["by"][i] += y[m].sum()
    a["br"] += float(((p - y) ** 2).sum())
def conf_up(c, pred, y):
    c["tp"] += int(np.logical_and(pred == 1, y == 1).sum()); c["fp"] += int(np.logical_and(pred == 1, y == 0).sum())
    c["fn"] += int(np.logical_and(pred == 0, y == 1).sum()); c["tn"] += int(np.logical_and(pred == 0, y == 0).sum())

fout = open("exp07_predictions.csv", "w")
fout.write("ImageId,PatientID,ViewPosition,HasPneumothorax,det_dice_tuned,det_dice_05,ens_dice_tuned,ens_dice_05,det_unc,ens_unc\n")
t0 = time.time()
for j, b in enumerate(te_ld):
    x = b["image"].to(device); yn = b["mask"].numpy()[0, 0]
    yb = (yn > 0).astype(np.uint8)
    with torch.no_grad():
        ps = [torch.sigmoid(nets[s](x))[0, 0].cpu().numpy() for s in (42, 43, 44)]
        em = np.stack(ps).mean(0)
        ee = entropy(em); me = np.stack([entropy(p) for p in ps]).mean(0)
        mi = np.clip(ee - me, 0, None); ed = entropy(ps[0])
    row = [te_df.iloc[j]["ImageId"], te_df.iloc[j]["PatientID"], te_df.iloc[j]["ViewPosition"],
           te_df.iloc[j]["HasPneumothorax"]]
    for key, prob, T, t in (("det", ps[0], T_det, t_det), ("ens", em, T_ens, t_ens)):
        eps = 1e-7
        pc = np.clip(prob, eps, 1 - eps)
        ps_ = 1 / (1 + np.exp(-np.log(pc / (1 - pc)) / T))
        for suf, tt in (("", t), ("05", 0.5)):
            kk = key if suf == "" else key + "05"
            acc_up(acc[kk], ps_, yb.astype(np.float32))
            conf_up(conf[kk], (ps_ >= tt).astype(np.uint8), yb)
        row.append(compute_segmentation_metrics(yb, ps_, threshold=t)["dice"])
        row.append(compute_segmentation_metrics(yb, ps_, threshold=0.5)["dice"])
    # row order: det_tuned, det_05, ens_tuned, ens_05
    fout.write(f"{row[0]},{row[1]},{row[2]},{row[3]},{row[4]},{row[5]},{row[6]},{row[7]},{topk(ed)},{topk(mi)}\n")
    if (j + 1) % 400 == 0:
        fout.flush(); print(f"{j+1}/{N} ({(time.time()-t0)/60:.1f}m)", flush=True)
fout.close()

def esce_of(a):
    t = a["bn"].sum()
    return float((a["bn"] / t * np.abs(a["bp"] / np.maximum(a["bn"], 1) - a["by"] / np.maximum(a["bn"], 1))).sum())
OUT = {"tuned": tuned,
       "esce": {k: round(esce_of(acc[k]), 6) for k in acc},
       "brier": {k: round(acc[k]["br"] / acc[k]["n"], 6) for k in acc}}
for k in conf:
    c = conf[k]; tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]
    OUT[k + "_pix"] = {"iou": round(tp / (tp + fp + fn), 4) if tp + fp + fn else 0.0,
                       "sens": round(tp / (tp + fn), 4) if tp + fn else 1.0,
                       "spec": round(tn / (tn + fp), 4) if tn + fp else 1.0}
df = pd.read_csv("exp07_predictions.csv")
for c in ["det_dice_tuned", "det_dice_05", "ens_dice_tuned", "ens_dice_05"]:
    OUT[c + "_mean"] = round(float(df[c].mean()), 4)
    OUT[c + "_posmean"] = round(float(df[df["HasPneumothorax"] == 1][c].mean()), 4)
rng = np.random.default_rng(42); n = len(df)
for c in ["det_dice_tuned", "ens_dice_tuned"]:
    v = df[c].values; bs = v[rng.integers(0, n, (1000, n))].mean(1)
    OUT[c + "_ci95"] = [round(float(np.percentile(bs, 2.5)), 4), round(float(np.percentile(bs, 97.5)), 4)]
OUT["wilcoxon_tuned"] = float(wilcoxon(df["ens_dice_tuned"], df["det_dice_tuned"], alternative="two-sided").pvalue)
for m, u in (("det_dice_tuned", "det_unc"), ("ens_dice_tuned", "ens_unc")):
    o = np.argsort(df[u].values, kind="stable"); d = df[m].values[o]
    cum = np.cumsum(d) / np.arange(1, n + 1); cov = np.arange(1, n + 1) / n
    OUT[m + "_aurc"] = round(float(np.trapezoid(1 - cum, cov)), 4)
    oo = np.argsort(-d, kind="stable"); dd = d[oo]
    cum_o = np.cumsum(dd) / np.arange(1, n + 1)
    OUT[m + "_eaurc"] = round(OUT[m + "_aurc"] - float(np.trapezoid(1 - cum_o, cov)), 4)
with open("exp07_metrics.json", "w") as f: json.dump(OUT, f, indent=2)
print(json.dumps(OUT, indent=2))
print("saved all")
